# Part C — Image-Based Product Matching

**Goal:** decide whether two listings are the same product using **only their pictures**.

```
Product image → preprocessing → pretrained network → embedding (a vector) → cosine similarity → ≥ threshold ? → match / no match
```

**What we do in this notebook**
1. **Baseline** — the `image_phash` column (a cheap "fingerprint" of the image, no deep learning).
2. **Experiment 1** — **ResNet50** pretrained on ImageNet (2048-number embedding).
3. **Experiment 2** — **CLIP ViT-B/32** pretrained on 400 million image–text pairs (512-number embedding).
4. Compare them, tune the threshold, look at **nearest neighbours**, and analyse **errors**.

We do **not** train any network — we only use pretrained ones as feature extractors (as the task suggests).

**External resources that influenced this notebook:** torchvision's pretrained ResNet50 (He et al., 2016, *Deep Residual Learning*);
OpenAI CLIP (Radford et al., 2021) loaded through the `sentence-transformers` library (`clip-ViT-B-32`); FAISS (Johnson et al., Facebook AI) for the optional fast-search demo;
the `image_phash` column provided by the Shopee dataset (perceptual hash, `imagehash` library).

⚠️ **Use a GPU** (Kaggle: Settings → Accelerator → GPU) and turn **Internet ON** (needed to download the pretrained weights).
On CPU, reduce `N_GROUPS` to ~500.

In [ ]:
# Install the extra library if the notebook environment does not have it yet
import importlib, subprocess, sys
for pkg, mod in [("sentence-transformers", "sentence_transformers")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

## 1. Setup — imports, settings and finding the dataset

This cell loads the libraries we need and finds the Shopee dataset automatically
(on Kaggle, Google Colab or your own computer — see `SETUP_GUIDE.md` for how to add the dataset).

In [ ]:
import os, re, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 100

SEED = 42                      # a fixed "random seed" so you get the same results every time you run
random.seed(SEED)
np.random.seed(SEED)

# ---------------- Settings you may want to change ----------------
N_GROUPS = 3000                # how many product groups to use (3000 groups ≈ 9,000 listings).
                               # Smaller = faster (use ~500 if you have no GPU). None = whole dataset (slow, needs lots of RAM).
RESULTS_DIR = "results"        # plots and tables are saved here
os.makedirs(RESULTS_DIR, exist_ok=True)


def find_data_dir():
    """Look for the folder that contains train.csv (works on Kaggle, Colab and a local computer)."""
    if os.environ.get("SHOPEE_DIR"):                       # you can force a path with this variable
        return os.environ["SHOPEE_DIR"]
    candidates = [
        "/kaggle/input/shopee-product-matching",           # Kaggle notebook (competition added via 'Add Input')
        "/kaggle/input/competitions/shopee-product-matching",
        "/content/shopee",                                 # Google Colab (see SETUP_GUIDE.md)
        "./shopee-product-matching", "./data", "../data", "../shopee-product-matching",
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, "train.csv")):
            return c
    for root, dirs, files in os.walk("/kaggle/input"):     # last resort: search everything under /kaggle/input
        if "train.csv" in files:
            return root
    raise FileNotFoundError("train.csv not found. Follow the steps in SETUP_GUIDE.md to add the dataset.")


DATA_DIR = find_data_dir()
IMG_DIR = os.path.join(DATA_DIR, "train_images")
print("Using data from:", DATA_DIR)

## 2. How we measure success (read this once — it is used in every experiment)

**The task, in one sentence:** for every listing, return the list of listings that are the *same product*.

We compare against the truth (`label_group`: same value = same product). For ONE listing:

| Symbol | Meaning |
|---|---|
| **returned** | how many listings our system says are the same product (it always includes the listing itself) |
| **true** | how many listings really are the same product (also includes itself) |
| **TP** | how many of the *returned* listings are really the same product |

* **Precision** = TP / returned  → "of the matches I claimed, how many were right?"
* **Recall** = TP / true  → "of the real matches, how many did I find?"
* **F1** = 2·TP / (returned + true)  → one number that balances precision and recall

The final score is the **average F1 over all listings**. This is exactly the metric of the original Kaggle competition.

**How a similarity score becomes a decision:** two listings are called the *same product* if `similarity ≥ threshold`.

**Why a "tune" pool and a "test" pool?**
We cannot choose the threshold on the same data we report the final score on — that would be cheating (the score would look
better than it really is). So we split the *product groups* into two pools:
* **tune pool (60 %)** — we try thresholds here and pick the best one,
* **test pool (40 %)** — we only *report* the score here, using the threshold chosen on the tune pool.

We split by *group* (not by listing) so that listings of one product never end up in both pools.

In [ ]:
# Thresholds we will try: 0.10, 0.12, ..., 1.00
THRESHOLDS = np.round(np.arange(0.10, 1.0001, 0.02), 2)


def sample_and_split(df, n_groups=N_GROUPS, test_frac=0.4, seed=SEED):
    """Pick `n_groups` random product groups and split those GROUPS into a 'tune' pool and a 'test' pool."""
    rng = np.random.RandomState(seed)
    groups = np.sort(df["label_group"].unique())
    if n_groups is not None and n_groups < len(groups):
        groups = rng.choice(groups, size=n_groups, replace=False)
    else:
        groups = groups.copy()
    rng.shuffle(groups)
    test_groups = groups[: int(len(groups) * test_frac)]
    sub = df[df["label_group"].isin(groups)].reset_index(drop=True)
    sub["split"] = np.where(sub["label_group"].isin(test_groups), "test", "tune")
    return sub


def sweep_thresholds(sim, labels, thresholds=THRESHOLDS):
    """
    sim    : N x N matrix, sim[i, j] = how similar listing i and listing j are (higher = more similar)
    labels : the true product group of each of the N listings
    For every threshold, return the average precision / recall / F1 over all listings,
    plus how many wrong matches (false positives) and missed matches (false negatives) there were.
    """
    same = labels[:, None] == labels[None, :]          # same[i, j] = True if i and j are truly the same product
    n_true = same.sum(axis=1)                          # number of true matches per listing (including itself)
    rows = []
    for t in thresholds:
        pred = sim >= t                                # our predicted matches
        np.fill_diagonal(pred, True)                   # a listing always matches itself
        n_pred = pred.sum(axis=1)                      # how many we returned per listing
        tp = (pred & same).sum(axis=1)                 # how many of those were correct
        rows.append({
            "threshold": t,
            "precision": float((tp / n_pred).mean()),
            "recall": float((tp / n_true).mean()),
            "f1": float((2 * tp / (n_pred + n_true)).mean()),
            "wrong_matches": int((n_pred - tp).sum()),      # false positives, counted per (listing, candidate)
            "missed_matches": int((n_true - tp).sum()),     # false negatives, counted per (listing, candidate)
        })
    return pd.DataFrame(rows)


def run_experiment(name, representation, similarity, sim_fn, sub, log, extra=None):
    """
    1) compute similarities inside the TUNE pool and pick the threshold with the best F1
    2) compute similarities inside the TEST pool and report the score at that threshold
    `sim_fn(idx)` must return the N x N similarity matrix for the listings `idx`.
    """
    t0 = time.time()
    labels = sub["label_group"].values
    tune_idx = np.where(sub["split"] == "tune")[0]
    test_idx = np.where(sub["split"] == "test")[0]
    sweep_tune = sweep_thresholds(sim_fn(tune_idx), labels[tune_idx])
    sweep_test = sweep_thresholds(sim_fn(test_idx), labels[test_idx])
    best = sweep_tune.loc[sweep_tune["f1"].idxmax()]
    thr = best["threshold"]
    at_thr = sweep_test[sweep_test["threshold"] == thr].iloc[0]
    row = {"Experiment": name, "Representation": representation, "Similarity": similarity,
           "Threshold": thr, "Tune F1": round(best["f1"], 4),
           "Test precision": round(at_thr["precision"], 4), "Test recall": round(at_thr["recall"], 4),
           "Test F1": round(at_thr["f1"], 4), "Seconds": round(time.time() - t0, 1)}
    if extra:
        row.update(extra)
    log.append(row)
    print(f"{name}: threshold={thr:.2f} | tune F1={best['f1']:.4f} | TEST F1={at_thr['f1']:.4f} "
          f"(precision {at_thr['precision']:.3f}, recall {at_thr['recall']:.3f})")
    return {"sweep_tune": sweep_tune, "sweep_test": sweep_test, "threshold": thr, "sim_fn": sim_fn}


def plot_threshold_curves(res, title, save_as=None):
    """Left: precision / recall / F1 versus threshold.  Right: number of wrong and missed matches."""
    st, se, thr = res["sweep_tune"], res["sweep_test"], res["threshold"]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(st["threshold"], st["precision"], label="precision (tune)", color="tab:blue")
    ax[0].plot(st["threshold"], st["recall"], label="recall (tune)", color="tab:orange")
    ax[0].plot(st["threshold"], st["f1"], label="F1 (tune)", color="tab:green", lw=2.5)
    ax[0].plot(se["threshold"], se["f1"], "--", label="F1 (test)", color="tab:red")
    ax[0].axvline(thr, color="black", ls=":", label=f"chosen threshold = {thr:.2f}")
    ax[0].set_xlabel("threshold"); ax[0].set_ylabel("score"); ax[0].set_title(title); ax[0].legend(loc="center left", fontsize=8)
    ax[1].plot(st["threshold"], st["wrong_matches"] + 1, label="wrong matches (false positives)", color="tab:red")
    ax[1].plot(st["threshold"], st["missed_matches"] + 1, label="missed matches (false negatives)", color="tab:purple")
    ax[1].axvline(thr, color="black", ls=":")
    ax[1].set_yscale("log"); ax[1].set_xlabel("threshold"); ax[1].set_ylabel("count (log scale)")
    ax[1].set_title("Errors vs threshold (tune pool)"); ax[1].legend()
    plt.tight_layout()
    if save_as:
        plt.savefig(os.path.join(RESULTS_DIR, save_as), bbox_inches="tight")
    plt.show()

## 3. Load the data and build the working subset

`train.csv` has ~34,000 listings. Comparing every listing with every other one needs a 34,000 × 34,000 table
(~4.7 GB), so we work with a random **subset of product groups** (`N_GROUPS`). All the real structure is kept
(each group is complete), only the number of "distractor" listings is smaller, so scores will be a little
*higher* than on the full dataset. We say this clearly in the README.

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
print("Full training file:", df.shape)

sub = sample_and_split(df, N_GROUPS)
labels_all = sub["label_group"].values
titles = sub["title"].fillna("").astype(str).to_numpy(dtype=object)
tune_idx = np.where(sub["split"] == "tune")[0]
test_idx = np.where(sub["split"] == "test")[0]
print(f"Working subset: {len(sub):,} listings from {sub['label_group'].nunique():,} groups")
print(f"  tune pool: {len(tune_idx):,} listings | test pool: {len(test_idx):,} listings")
sub.head()

## 4. What is an embedding? (read this first)

A neural network that recognises objects works in layers. The last layers decide "this is a shoe", but just *before* that decision the network holds a
list of numbers that summarises **what is in the picture** (shapes, textures, colours, object type). That list is the **embedding**.

* Images that look / mean the same → embeddings **point in similar directions**.
* So we can find similar images by **comparing vectors**: this is a *similarity search*.
* **Cosine similarity** measures the angle between two vectors: 1 = same direction, 0 = unrelated. We *normalise* every embedding to length 1,
  so cosine similarity = dot product (a very fast operation that works on thousands of images at once with one matrix multiplication).
* **From similarity to a decision:** `similarity ≥ threshold → "same product"`. The threshold is tuned on the tune pool, exactly like in Part B.

A 2-number toy example (real embeddings have 512 or 2048 numbers):

In [ ]:
a = np.array([1.0, 0.0]);  b = np.array([0.9, 0.1]);  c = np.array([0.0, 1.0])
unit = lambda v: v / np.linalg.norm(v)
print("cosine(a, b) =", round(float(unit(a) @ unit(b)), 3), " → almost the same direction → similar")
print("cosine(a, c) =", round(float(unit(a) @ unit(c)), 3), " → perpendicular → unrelated")

## 5. Image preprocessing

Networks need a **fixed input size** (224 × 224) and numbers in the range they were trained on. Steps we use:
1. open the file and convert to RGB (some files are grey-scale or have transparency),
2. shrink very large photos (max 512 px) to save time and memory,
3. **ResNet:** squash the picture to 224 × 224 (keeps the whole product, slightly distorts the shape) and normalise with the ImageNet mean/std;
4. **CLIP:** its own preprocessing = resize the short side to 224, then **centre-crop** 224 × 224 (may cut off the sides of non-square images).

Let us look at what the two networks actually "see":

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from PIL import ImageFile
import torch
from torchvision import models, transforms
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True           # do not crash on slightly damaged files
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)


def load_one(fname):
    """Open one image, make it RGB and shrink it if it is huge."""
    img = Image.open(os.path.join(IMG_DIR, fname)).convert("RGB")
    img.thumbnail((512, 512))
    return img


def batches(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]


squash_224 = transforms.Resize((224, 224))                                         # what we do for ResNet
clip_style = transforms.Compose([transforms.Resize(224), transforms.CenterCrop(224)])  # what CLIP does

demo_files = sub["image"].sample(4, random_state=3).tolist()
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for c, f in enumerate(demo_files):
    im = load_one(f)
    axes[0, c].imshow(im);  axes[0, c].set_title(f"original {im.size}", fontsize=9)
    axes[1, c].imshow(squash_224(im));  axes[1, c].set_title("ResNet input (squashed 224x224)", fontsize=9)
    axes[2, c].imshow(clip_style(im));  axes[2, c].set_title("CLIP input (centre crop 224x224)", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "C_preprocessing.png"), bbox_inches="tight")
plt.show()

**What it shows:** squashing keeps everything but distorts proportions; the CLIP-style crop keeps proportions but can lose the edges of wide or tall photos.
Neither is perfect — one source of the errors we analyse later.

In [ ]:
results = []        # one row per experiment
experiments = {}    # thresholds, curves and embeddings of every experiment
embeddings = {}     # name -> (N, dim) array of L2-normalised embeddings for the whole subset
emb_seconds = {}    # how long computing each embedding took

## 6. Baseline — perceptual hash (`image_phash`), no deep learning

**What it is:** a 64-bit fingerprint computed from the *low-frequency structure* of the image. Two images that look nearly identical have hashes that differ in only a few bits.
The number of differing bits is the **Hamming distance** (0 = identical, ~32 = unrelated).

**How we use it with the same code as the neural networks:** turn each hash into a vector of 64 numbers that are +1 / −1 (bit = 1 / 0) and divide by 8 (so its length is 1).
Then `cosine similarity = 1 − Hamming/32` — identical hashes give 1.0, unrelated ones about 0.

**Why start here:** it is free (already in the CSV), instant, and tells us how much a *deep* model has to improve upon.

In [ ]:
def phash_to_vectors(hashes):
    """'e9b9d5b0a4a4b6b5' -> 64 numbers (+1/-1), scaled to length 1."""
    bits = np.array([[int(b) for b in format(int(h, 16), "064b")] for h in hashes], dtype=np.float32)
    return (bits * 2 - 1) / 8.0


embeddings["Baseline: phash"] = phash_to_vectors(sub["image_phash"])
emb_seconds["Baseline: phash"] = 0.0
E = embeddings["Baseline: phash"]

experiments["Baseline: phash"] = run_experiment(
    "Baseline: phash", "64-bit perceptual hash", "Cosine (= 1 - Hamming/32)", lambda idx: E[idx] @ E[idx].T, sub, results,
    extra={"Dim": E.shape[1]})

# the strictest possible phash rule: "identical hash only" is simply the threshold = 1.0 row of the sweep
exact = experiments["Baseline: phash"]["sweep_test"].query("threshold == 1.0").iloc[0]
print(f"\nRule 'identical phash only' on the test pool: precision {exact['precision']:.3f}, recall {exact['recall']:.3f}, F1 {exact['f1']:.3f}")

## 7. Experiment 1 — ResNet50 (ImageNet-pretrained)

**Hypothesis:** a network trained to recognise 1,000 object classes extracts features about shape, texture and colour that describe *what kind of product* is shown,
so different photos of the same product should land close together — better than a hash which only looks at coarse structure.

**How:** load `torchvision`'s ResNet50, replace its final classification layer (`fc`) with "do nothing" (`Identity`) so the output is the
**2048-number** vector produced by the last pooling layer, then L2-normalise.

In [ ]:
def embed_resnet(fnames, batch_size=64):
    out = []
    with ThreadPoolExecutor(8) as pool:                      # open images with several threads while the GPU computes
        for chunk in tqdm(list(batches(list(fnames), batch_size)), desc="ResNet50"):
            imgs = list(pool.map(load_one, chunk))
            x = torch.stack([resnet_tf(im) for im in imgs]).to(device)
            with torch.no_grad():                            # no training → no gradients → faster, less memory
                feats = resnet(x)
            out.append(feats.cpu().numpy())
    emb = np.concatenate(out).astype(np.float32)
    return emb / np.linalg.norm(emb, axis=1, keepdims=True)   # length 1 → cosine = dot product


cache_file = os.path.join(RESULTS_DIR, f"img_emb_resnet50_{len(sub)}.npy")
if os.path.exists(cache_file) and np.load(cache_file).shape[0] == len(sub):
    embeddings["Exp 1: ResNet50"] = np.load(cache_file)
    emb_seconds["Exp 1: ResNet50"] = float("nan")
    print("Loaded cached embeddings", embeddings["Exp 1: ResNet50"].shape)
else:
    resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    resnet.fc = torch.nn.Identity()
    resnet = resnet.to(device).eval()
    resnet_tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),     # ImageNet mean / std
    ])
    t0 = time.time()
    embeddings["Exp 1: ResNet50"] = embed_resnet(sub["image"])
    emb_seconds["Exp 1: ResNet50"] = time.time() - t0
    np.save(cache_file, embeddings["Exp 1: ResNet50"])
    print(f"Embedded {len(sub):,} images in {emb_seconds['Exp 1: ResNet50']:.0f}s -> {embeddings['Exp 1: ResNet50'].shape}")

E = embeddings["Exp 1: ResNet50"]
experiments["Exp 1: ResNet50"] = run_experiment(
    "Exp 1: ResNet50", "ResNet50 (ImageNet), global-avg-pool", "Cosine", lambda idx: E[idx] @ E[idx].T, sub, results,
    extra={"Dim": E.shape[1]})

## 8. Experiment 2 — CLIP ViT-B/32

**Hypothesis:** CLIP was trained to match images with their *text descriptions* (400 M web image–caption pairs), so its embedding captures *what the object is*
(semantics), not only textures. That should help when the same product is photographed in different ways, and it should beat ResNet.

**How:** `SentenceTransformer("clip-ViT-B-32")` accepts PIL images and returns a **512-number** embedding. Different model *and* different embedding size than Experiment 1.

In [ ]:
from sentence_transformers import SentenceTransformer


def embed_clip(fnames, batch_size=64):
    out = []
    with ThreadPoolExecutor(8) as pool:
        for chunk in tqdm(list(batches(list(fnames), batch_size)), desc="CLIP"):
            imgs = list(pool.map(load_one, chunk))
            e = clip_model.encode(imgs, batch_size=batch_size, convert_to_numpy=True,
                                  normalize_embeddings=True, show_progress_bar=False)
            out.append(e)
    return np.concatenate(out).astype(np.float32)


cache_file = os.path.join(RESULTS_DIR, f"img_emb_clip_{len(sub)}.npy")
if os.path.exists(cache_file) and np.load(cache_file).shape[0] == len(sub):
    embeddings["Exp 2: CLIP ViT-B/32"] = np.load(cache_file)
    emb_seconds["Exp 2: CLIP ViT-B/32"] = float("nan")
    print("Loaded cached embeddings", embeddings["Exp 2: CLIP ViT-B/32"].shape)
else:
    clip_model = SentenceTransformer("clip-ViT-B-32", device=device)
    t0 = time.time()
    embeddings["Exp 2: CLIP ViT-B/32"] = embed_clip(sub["image"])
    emb_seconds["Exp 2: CLIP ViT-B/32"] = time.time() - t0
    np.save(cache_file, embeddings["Exp 2: CLIP ViT-B/32"])
    print(f"Embedded {len(sub):,} images in {emb_seconds['Exp 2: CLIP ViT-B/32']:.0f}s -> {embeddings['Exp 2: CLIP ViT-B/32'].shape}")

E = embeddings["Exp 2: CLIP ViT-B/32"]
experiments["Exp 2: CLIP ViT-B/32"] = run_experiment(
    "Exp 2: CLIP ViT-B/32", "CLIP ViT-B/32 image encoder", "Cosine", lambda idx: E[idx] @ E[idx].T, sub, results,
    extra={"Dim": E.shape[1]})

## 9. Retrieval quality (nearest neighbours) and the results table

Besides F1 we also measure plain *retrieval*: for every test listing, find its **single most similar other listing** and check if it is really the same product.
* **Hit@1** = share of listings whose #1 neighbour is a true match.
* **Hit@5** = share of listings that have at least one true match among their top-5 neighbours.

To find neighbours **without building the huge N × N table**, we process the listings in chunks and keep only the top-k per row (this is also the answer to the computational question in section 12).

In [ ]:
def top_k_neighbors(emb, k=5, chunk=1000):
    """For every row of `emb`, return the indices and similarities of its k most similar OTHER rows."""
    n = len(emb)
    nbr_idx = np.zeros((n, k), dtype=np.int64)
    nbr_sim = np.zeros((n, k), dtype=np.float32)
    for s in range(0, n, chunk):
        sims = emb[s:s + chunk] @ emb.T                                   # (chunk, n) — never the full n x n
        sims[np.arange(sims.shape[0]), np.arange(s, s + sims.shape[0])] = -np.inf   # a listing is not its own neighbour
        top = np.argpartition(-sims, k, axis=1)[:, :k]                    # the k best (unordered)
        top_sims = np.take_along_axis(sims, top, axis=1)
        order = np.argsort(-top_sims, axis=1)                             # sort them best-first
        nbr_idx[s:s + chunk] = np.take_along_axis(top, order, axis=1)
        nbr_sim[s:s + chunk] = np.take_along_axis(top_sims, order, axis=1)
    return nbr_idx, nbr_sim


lab_test = labels_all[test_idx]
neighbors = {}
for i_, name in enumerate(experiments):
    emb_pool = embeddings[name][test_idx]
    nbr_idx, nbr_sim = top_k_neighbors(emb_pool, k=5)
    neighbors[name] = (nbr_idx, nbr_sim)
    is_true = lab_test[nbr_idx] == lab_test[:, None]                      # (n, 5) True where the neighbour is the same product
    results[i_]["Hit@1"] = round(float(is_true[:, 0].mean()), 4)
    results[i_]["Hit@5"] = round(float(is_true.any(axis=1).mean()), 4)
    results[i_]["Embedding MB"] = round(embeddings[name].nbytes / 1e6, 1)
    results[i_]["Embedding seconds"] = round(emb_seconds[name], 1)

results_df = pd.DataFrame(results)
display(results_df[["Experiment", "Representation", "Dim", "Similarity", "Threshold", "Tune F1", "Test precision",
                    "Test recall", "Test F1", "Hit@1", "Hit@5", "Embedding MB", "Embedding seconds"]])
results_df.to_csv(os.path.join(RESULTS_DIR, "part_c_results.csv"), index=False)

plt.figure(figsize=(9, 3.8))
plt.barh(results_df["Experiment"][::-1], results_df["Test F1"][::-1], color="tab:green")
for k_, v in enumerate(results_df["Test F1"][::-1]):
    plt.text(v, k_, f" {v:.3f}", va="center")
plt.xlim(0, 1); plt.xlabel("Test F1"); plt.title("Image-only matching: test F1 by method")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "C_results_bar.png"), bbox_inches="tight"); plt.show()

best_name = results_df.loc[results_df["Tune F1"].idxmax(), "Experiment"]
print("Best method chosen on the TUNE pool:", best_name)

**How to read it:** compare the three rows on *Test F1* and on *Hit@1*. Note that Hit@1 only checks the single closest neighbour (no threshold), while F1 also needs a good threshold.
`Embedding seconds` (blank = loaded from cache) shows the compute cost; `Embedding MB` shows the memory cost of storing the embeddings of our subset.

## 10. Which similarity measure? Cosine vs dot product vs Euclidean distance

For **length-1 vectors** the three measures carry exactly the same information:
* cosine similarity = dot product (because both lengths are 1);
* squared Euclidean distance = 2 − 2·cosine. So *small distance ⇔ large cosine* and the neighbour **ranking is identical**.

We verify it numerically on CLIP embeddings:

In [ ]:
from scipy.spatial.distance import cdist

sample = embeddings["Exp 2: CLIP ViT-B/32"][test_idx][:500]
cos = sample @ sample.T
dist = cdist(sample, sample)
print("squared Euclidean distance == 2 - 2*cosine ?", bool(np.allclose(dist ** 2, 2 - 2 * cos, atol=1e-4)))
np.fill_diagonal(cos, -np.inf)
np.fill_diagonal(dist, np.inf)
print("Same nearest neighbour for every image with cosine and Euclidean ?", bool((cos.argmax(1) == dist.argmin(1)).all()))

**Conclusion:** because we normalise, the choice between these three does **not** change the ranking — we use cosine (= dot product) because it is the cheapest (one matrix multiplication).
*Without* normalisation they would differ: the dot product would favour images whose embedding has a large length, which is usually unrelated to product identity.

## 11. Threshold analysis

In [ ]:
for name in experiments:
    plot_threshold_curves(experiments[name], name, save_as=f"C_threshold_{name.split(':')[0].replace(' ', '')}.png")

In [ ]:
plt.figure(figsize=(7, 5))
for name, res in experiments.items():
    sw = res["sweep_tune"]
    plt.plot(sw["recall"], sw["precision"], label=name)
plt.xlabel("recall"); plt.ylabel("precision"); plt.title("Precision-recall trade-off (each point = one threshold)")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(RESULTS_DIR, "C_precision_recall.png"), bbox_inches="tight")
plt.show()

sw = experiments[best_name]["sweep_tune"]
ops = {"Max F1 (our choice)": sw.loc[sw["f1"].idxmax()]}
hi_p = sw[sw["precision"] >= 0.9]
if len(hi_p):
    ops["Safest: precision >= 0.9"] = hi_p.loc[hi_p["recall"].idxmax()]
hi_r = sw[sw["recall"] >= 0.9]
if len(hi_r):
    ops["Widest net: recall >= 0.9"] = hi_r.loc[hi_r["precision"].idxmax()]
print(f"Operating points for {best_name}:")
display(pd.DataFrame(ops).T[["threshold", "precision", "recall", "f1", "wrong_matches", "missed_matches"]].round(3))

**How to read the plots:**
* A **low** threshold accepts weakly similar images: high recall, but many wrong matches (visually similar but different products).
* A **high** threshold only accepts near-identical images: high precision, but many true matches are missed (same product, different photo).
* Different models need different thresholds: ResNet features are all positive so even unrelated images have a high cosine (the best threshold is high);
  the phash vectors are spread around 0. **A threshold is never transferable between models** — always re-tune it.
* We choose the threshold with the best F1 on the tune pool and report on the test pool (dashed red curve shows that the choice generalises).

## 12. Nearest-neighbour analysis

For a few **query images** from the test pool we retrieve the 5 most similar images. Green frame = really the same product, red frame = different product.
The number is the cosine similarity.

In [ ]:
def show_neighbors(query_positions, name, k=5, fname=None):
    """query_positions: positions inside the test pool. Draws the query and its k neighbours (green = same product)."""
    nbr_idx, nbr_sim = neighbors[name]
    fig, axes = plt.subplots(len(query_positions), k + 1, figsize=((k + 1) * 2.4, len(query_positions) * 2.8))
    axes = np.array(axes).reshape(len(query_positions), k + 1)
    for r, q in enumerate(query_positions):
        row_q = sub.iloc[test_idx[q]]
        axes[r, 0].imshow(load_one(row_q["image"]))
        axes[r, 0].set_title("QUERY\n" + short_title(row_q["title"], 22, 44), fontsize=7)
        for c in range(k):
            j = nbr_idx[q, c]
            row_j = sub.iloc[test_idx[j]]
            ok = row_j["label_group"] == row_q["label_group"]
            axes[r, c + 1].imshow(load_one(row_j["image"]))
            axes[r, c + 1].set_title(f"#{c + 1}  sim {nbr_sim[q, c]:.2f}  {'✓' if ok else '✗'}\n" + short_title(row_j["title"], 22, 44),
                                     fontsize=7, color="darkgreen" if ok else "darkred")
            for sp in axes[r, c + 1].spines.values():
                sp.set_edgecolor("green" if ok else "red"); sp.set_linewidth(4)
        for ax in axes[r]:
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"Nearest neighbours — {name}", y=1.0)
    plt.tight_layout()
    if fname:
        plt.savefig(os.path.join(RESULTS_DIR, fname), bbox_inches="tight")
    plt.show()


import textwrap


def short_title(s, width=30, max_chars=80):
    return textwrap.fill(textwrap.shorten(str(s), max_chars, placeholder="…"), width)


rng = np.random.RandomState(7)
random_queries = rng.choice(len(test_idx), size=5, replace=False)
show_neighbors(random_queries, "Exp 2: CLIP ViT-B/32", fname="C_neighbors_clip_random.png")

In [ ]:
# The same queries with ResNet50: does the weaker model retrieve different (worse) neighbours?
show_neighbors(random_queries[:3], "Exp 1: ResNet50", fname="C_neighbors_resnet.png")

In [ ]:
# Queries where the BEST model's nearest neighbour is WRONG — these show where it fails
nbr_idx, nbr_sim = neighbors[best_name]
wrong_top1 = np.where(lab_test[nbr_idx[:, 0]] != lab_test)[0]
print(f"{best_name}: the #1 neighbour is wrong for {len(wrong_top1)} of {len(lab_test)} test queries ({100 * len(wrong_top1) / len(lab_test):.1f} %)")
if len(wrong_top1):
    show_neighbors(np.random.RandomState(1).choice(wrong_top1, size=min(4, len(wrong_top1)), replace=False), best_name,
                   fname="C_neighbors_failures.png")

**What to look for:** do the green neighbours look like the same *product* or just the same *picture style*?
Do the red ones look like the same product *family* (another colour / size)? Which queries have **no** green neighbour even though
the dataset contains a true partner — what is different about those photos?

## 13. Computational considerations

* **Memory of the embeddings** is tiny: 34,250 × 512 floats ≈ 70 MB (CLIP) or ≈ 280 MB (ResNet, 2048 numbers).
* **Memory of the full similarity table** is the problem: N × N × 4 bytes. For the whole dataset that is 34,250² × 4 ≈ **4.7 GB**, and it grows with the *square* of N
  (a catalogue of 1 million products would need 4 **terabytes**).
* **Solution:** never build the full table. Compute similarities **in chunks** and keep only the top-k per row (our `top_k_neighbors`), or use an
  **approximate nearest-neighbour index** such as FAISS (HNSW / IVF), which finds neighbours in a fraction of the time with a small loss in accuracy.
* **Cost of the embedding itself** (running the network on every image) is the other big cost — it is paid once per image, and the GPU matters a lot (see the `Embedding seconds` column).

In [ ]:
n_full = 34250
print(f"Embeddings for the whole dataset: CLIP {n_full * 512 * 4 / 1e6:.0f} MB | ResNet50 {n_full * 2048 * 4 / 1e6:.0f} MB")
print(f"Full similarity table for the whole dataset: {n_full ** 2 * 4 / 1e9:.1f} GB")

emb_pool = embeddings["Exp 2: CLIP ViT-B/32"][test_idx]
t0 = time.time(); _ = emb_pool @ emb_pool.T; t_full = time.time() - t0
t0 = time.time(); idx_chunk, _ = top_k_neighbors(emb_pool, k=5, chunk=500); t_chunk = time.time() - t0
print(f"\nTest pool ({len(emb_pool):,} images): full similarity table {t_full:.3f}s ({len(emb_pool) ** 2 * 4 / 1e6:.0f} MB)"
      f" | chunked top-5 search {t_chunk:.3f}s")

try:
    import faiss
    index = faiss.IndexFlatIP(emb_pool.shape[1])          # exact inner-product search (use IndexHNSWFlat for approximate)
    index.add(emb_pool)
    t0 = time.time(); D, I = index.search(emb_pool, 6); t_faiss = time.time() - t0   # 6 = itself + 5 neighbours
    agree = np.mean(I[:, 1] == idx_chunk[:, 0])
    print(f"FAISS exact search: {t_faiss:.3f}s | its #1 neighbour agrees with ours for {100 * agree:.1f} % of the images")
except ImportError:
    print("FAISS is not installed (optional) — try:  pip install faiss-cpu")

## 14. Error analysis (best image model, test pool)

* **Wrong match (false positive):** different products, but the images look alike → similarity ≥ threshold.
* **Missed match (false negative):** same product, but the images look different → similarity < threshold.

In [ ]:
chosen_thr = experiments[best_name]["threshold"]
E_best = embeddings[best_name][test_idx]
sim_test = E_best @ E_best.T
n_t = len(test_idx)
same_t = lab_test[:, None] == lab_test[None, :]
upper_t = np.triu(np.ones((n_t, n_t), dtype=bool), k=1)
pred_t = sim_test >= chosen_thr
fp_i, fp_j = np.where(upper_t & ~same_t & pred_t)
fn_i, fn_j = np.where(upper_t & same_t & ~pred_t)
print(f"{best_name} @ threshold {chosen_thr:.2f}: wrong matches = {len(fp_i):,} | missed matches = {len(fn_i):,}")


def show_image_pairs(ii, jj, sims, heading):
    """Show pairs of listings: [image A | image B] with similarity and titles."""
    n_pairs = len(ii)
    fig, axes = plt.subplots(n_pairs, 2, figsize=(5.6, 3.0 * n_pairs))
    axes = np.array(axes).reshape(n_pairs, 2)
    for r in range(n_pairs):
        for c, idx in enumerate((ii[r], jj[r])):
            row = sub.iloc[test_idx[idx]]
            axes[r, c].imshow(load_one(row["image"]))
            axes[r, c].set_title(short_title(row["title"], 30, 70), fontsize=7)
            axes[r, c].axis("off")
        axes[r, 0].text(1.0, -0.02, f"similarity {sims[r]:.3f}", transform=axes[r, 0].transAxes, ha="center", va="top", fontsize=9, color="navy")
    fig.suptitle(heading, y=1.0)
    plt.tight_layout()
    plt.show()

In [ ]:
# ❌ WRONG matches: the most confident ones (random 6 from the 40 highest similarities, to avoid cherry-picking)
if len(fp_i):
    s = sim_test[fp_i, fp_j]
    top = np.argsort(-s)[:40]
    pick = np.random.RandomState(0).choice(top, size=min(6, len(top)), replace=False)
    show_image_pairs(fp_i[pick], fp_j[pick], s[pick], "WRONG matches: different products that look alike")

In [ ]:
# ❌ MISSED matches: random 6 from the 40 lowest similarities
if len(fn_i):
    s = sim_test[fn_i, fn_j]
    low = np.argsort(s)[:40]
    pick = np.random.RandomState(0).choice(low, size=min(6, len(low)), replace=False)
    show_image_pairs(fn_i[pick], fn_j[pick], s[pick], "MISSED matches: same product, very different images")

**Sort what you see into these categories** (write the counts/examples in your observations):
* *same product photographed differently* (angle, zoom, background, collage, with / without packaging) → missed matches;
* *different colour / size / flavour variant of the same item or same stock photo* → wrong matches;
* *different packaging*, *cropped images*, *low-quality / tiny images*, *text overlay or watermark* → both kinds of errors.

In [ ]:
# Can the TITLE rescue these image errors?  (a preview of the Finale)
from sklearn.feature_extraction.text import TfidfVectorizer

X_t = TfidfVectorizer().fit_transform(titles)


def title_sim(i, j):
    return float(X_t[test_idx[i]].multiply(X_t[test_idx[j]]).sum())


def median_title_sim(ii, jj, k=300):
    pick = np.random.RandomState(0).choice(len(ii), size=min(k, len(ii)), replace=False) if len(ii) else []
    vals = [title_sim(ii[p], jj[p]) for p in pick]
    return float(np.median(vals)) if vals else float("nan")


print("Median TITLE similarity (TF-IDF cosine):")
print(f"  image says 'match' but products DIFFER (wrong matches): {median_title_sim(fp_i, fp_j):.2f}")
print(f"  image says 'no match' but products are the SAME (missed matches): {median_title_sim(fn_i, fn_j):.2f}")
print("If titles of the MISSED matches are quite similar, the title can rescue them; "
      "if titles of the WRONG matches are clearly different, the title can veto them.")

## 15. Answers to the guiding questions

* **What does an embedding capture?** A compact summary of the picture's content: for ResNet mainly shapes/textures/colours of ImageNet-like objects;
  for CLIP also the *semantic* meaning (what the object is), because it was trained with text.
* **Why can two images of the same product have different embeddings?** Different angle, crop, zoom, background, lighting, packaging, text overlay, or a completely different photo.
  The network encodes the *picture*, not the product's identity.
* **Why can two different products have almost the same embedding?** Same category and style (white background, same shape), colour/size variants, stock photos —
  the visible differences are tiny compared with what they share.
* **Which similarity metric works best?** For normalised embeddings cosine, dot product and Euclidean distance give the same ranking (section 10); we use cosine.
* **How does the threshold affect results?** Low → many wrong matches (high recall, low precision); high → many missed matches. See section 11.
* **Computational challenges?** The N × N similarity table grows quadratically (section 13): use chunking or ANN search (FAISS); the embedding step is the GPU-heavy part.

## 16. My observations and conclusions  ✍️ (write these yourself after running)

1. **Baseline (phash):** test F1 = … ; the rule "identical hash only" gives precision … / recall … → meaning: …
2. **ResNet50 vs CLIP:** CLIP / ResNet is better by … F1 points. My explanation: …
3. **Embedding size / speed trade-off:** … (use the table: dimension, seconds, MB)
4. **Nearest neighbours:** Hit@1 = … for the best model. In the pictures the neighbours that are right look like …, the wrong ones look like …
5. **Threshold:** I chose … ; at higher thresholds … at lower thresholds …
6. **Typical false positives:** … (copy 1–2 real examples with the category: variant / same stock photo / similar packaging …)
7. **Typical false negatives:** … (category: different angle / background / collage …)
8. **Can images alone do the job?** … (my answer, backed by the numbers)
9. **What the image model cannot see but the title can:** … (from the last cell: median title similarity of the errors)